# 🎙️ Gender Recognition by Voice

**Tujuan:** Mengidentifikasi gender (Male/Female) berdasarkan fitur suara.

**Analisis Perbandingan:**
1. MFCC saja
2. Pitch saja
3. MFCC + Pitch

**Model:** Logistic Regression, SVM, Random Forest, CNN

In [ ]:
import sys
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

# Import model modules
sys.path.insert(0, os.path.abspath('.'))
from models.logistic_model import train_model as train_logistic, evaluate_model as eval_logistic
from models.svm_model import train_model as train_svm, evaluate_model as eval_svm
from models.rf_model import train_model as train_rf, evaluate_model as eval_rf

print("All imports successful!")

## 1. Load Dataset

In [ ]:
df = pd.read_csv('dataset_fitur_audio.csv')
print(f"Shape: {df.shape}")
print(f"\nLabel distribution:\n{df['label'].value_counts()}")
df.head()

## 2. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Label distribution
colors = ['#4ECDC4', '#FF6B6B']
df['label'].value_counts().plot(kind='bar', ax=axes[0], color=colors, edgecolor='black')
axes[0].set_title('Distribusi Gender', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Gender')
axes[0].set_ylabel('Jumlah')

# Pitch comparison
male_pitch = df[df['label'] == 'male']['pitch_mean']
female_pitch = df[df['label'] == 'female']['pitch_mean']
axes[1].hist(male_pitch, bins=40, alpha=0.7, label='Male', color='#4ECDC4', edgecolor='black')
axes[1].hist(female_pitch, bins=40, alpha=0.7, label='Female', color='#FF6B6B', edgecolor='black')
axes[1].set_title('Distribusi Pitch Mean: Male vs Female', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Pitch Mean (Hz)')
axes[1].set_ylabel('Frekuensi')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Male   - Pitch Mean: {male_pitch.mean():.2f} Hz (std: {male_pitch.std():.2f})")
print(f"Female - Pitch Mean: {female_pitch.mean():.2f} Hz (std: {female_pitch.std():.2f})")

## 3. Definisi Kolom Fitur

In [ ]:
# Kolom MFCC (13 MFCC x 4 statistik = 52 fitur)
mfcc_cols = []
for i in range(1, 14):
    for stat in ['mean', 'std', 'min', 'max']:
        mfcc_cols.append(f'mfcc_{i}_{stat}')

# Kolom Pitch (4 fitur)
pitch_cols = ['pitch_mean', 'pitch_std', 'pitch_min', 'pitch_max']

# Kolom MFCC + Pitch (56 fitur)
mfcc_pitch_cols = mfcc_cols + pitch_cols

print(f"Jumlah fitur MFCC       : {len(mfcc_cols)}")
print(f"Jumlah fitur Pitch      : {len(pitch_cols)}")
print(f"Jumlah fitur MFCC+Pitch : {len(mfcc_pitch_cols)}")

## 4. Data Splitting (Train/Test)

In [ ]:
y = df['label']

# 3 variasi fitur
X_mfcc = df[mfcc_cols]
X_pitch = df[pitch_cols]
X_mfcc_pitch = df[mfcc_pitch_cols]

# Split 80:20 dengan stratify
X_mfcc_train, X_mfcc_test, y_train, y_test = train_test_split(
    X_mfcc, y, test_size=0.2, random_state=42, stratify=y)

X_pitch_train, X_pitch_test, _, _ = train_test_split(
    X_pitch, y, test_size=0.2, random_state=42, stratify=y)

X_mp_train, X_mp_test, _, _ = train_test_split(
    X_mfcc_pitch, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set : {len(y_train)} samples")
print(f"Testing set  : {len(y_test)} samples")

## 5. Training & Evaluasi - 9 Kombinasi Eksperimen

| No | Model | Fitur |
|----|-------|-------|
| 1 | Logistic Regression | MFCC |
| 2 | Logistic Regression | Pitch |
| 3 | Logistic Regression | MFCC + Pitch |
| 4 | SVM | MFCC |
| 5 | SVM | Pitch |
| 6 | SVM | MFCC + Pitch |
| 7 | Random Forest | MFCC |
| 8 | Random Forest | Pitch |
| 9 | Random Forest | MFCC + Pitch |

In [ ]:
# Konfigurasi eksperimen
experiments = {
    'MFCC':       (X_mfcc_train, X_mfcc_test),
    'Pitch':      (X_pitch_train, X_pitch_test),
    'MFCC+Pitch': (X_mp_train, X_mp_test),
}

models_config = {
    'Logistic Regression': (train_logistic, eval_logistic),
    'SVM':                 (train_svm, eval_svm),
    'Random Forest':       (train_rf, eval_rf),
}

# Jalankan semua eksperimen
results_list = []

for feat_name, (X_tr, X_te) in experiments.items():
    for model_name, (train_fn, eval_fn) in models_config.items():
        print(f"Training {model_name} with {feat_name}...", end=" ")
        model = train_fn(X_tr, y_train)
        res = eval_fn(model, X_te, y_test)
        results_list.append({
            'Model': model_name,
            'Fitur': feat_name,
            'Accuracy': res['accuracy'],
            'Precision': res['precision'],
            'Recall': res['recall'],
            'F1-Score': res['f1'],
        })
        print(f"Accuracy: {res['accuracy']:.4f}")

print("\nSemua eksperimen selesai!")

## 6. Tabel Komparasi Hasil

In [ ]:
results_df = pd.DataFrame(results_list)

# Format angka
styled = results_df.style.format({
    'Accuracy': '{:.4f}',
    'Precision': '{:.4f}',
    'Recall': '{:.4f}',
    'F1-Score': '{:.4f}'
}).background_gradient(subset=['Accuracy', 'F1-Score'], cmap='YlGn')

styled

## 7. Visualisasi Perbandingan Akurasi

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))

feature_types = ['MFCC', 'Pitch', 'MFCC+Pitch']
model_names = ['Logistic Regression', 'SVM', 'Random Forest']
colors_map = {'Logistic Regression': '#4ECDC4', 'SVM': '#45B7D1', 'Random Forest': '#96CEB4'}

x = np.arange(len(feature_types))
width = 0.25

for i, model_name in enumerate(model_names):
    subset = results_df[results_df['Model'] == model_name]
    accuracies = [subset[subset['Fitur'] == ft]['Accuracy'].values[0] for ft in feature_types]
    bars = ax.bar(x + i * width, accuracies, width, label=model_name,
                  color=colors_map[model_name], edgecolor='black', linewidth=0.5)
    for bar, acc in zip(bars, accuracies):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
                f'{acc:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_xlabel('Variasi Fitur', fontsize=13)
ax.set_ylabel('Accuracy', fontsize=13)
ax.set_title('Perbandingan Akurasi: Model x Fitur', fontsize=16, fontweight='bold')
ax.set_xticks(x + width)
ax.set_xticklabels(feature_types, fontsize=12)
ax.legend(fontsize=11)
ax.set_ylim(0, 1.1)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Visualisasi Perbandingan F1-Score

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))

for i, model_name in enumerate(model_names):
    subset = results_df[results_df['Model'] == model_name]
    f1_scores = [subset[subset['Fitur'] == ft]['F1-Score'].values[0] for ft in feature_types]
    bars = ax.bar(x + i * width, f1_scores, width, label=model_name,
                  color=colors_map[model_name], edgecolor='black', linewidth=0.5)
    for bar, f1 in zip(bars, f1_scores):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
                f'{f1:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_xlabel('Variasi Fitur', fontsize=13)
ax.set_ylabel('F1-Score', fontsize=13)
ax.set_title('Perbandingan F1-Score: Model x Fitur', fontsize=16, fontweight='bold')
ax.set_xticks(x + width)
ax.set_xticklabels(feature_types, fontsize=12)
ax.legend(fontsize=11)
ax.set_ylim(0, 1.1)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Analisis: Apakah Pitch Benar-Benar Krusial?

Kita bandingkan performa **Pitch saja** vs **MFCC saja** vs **MFCC+Pitch** untuk membuktikan kontribusi Pitch.

In [ ]:
# Heatmap performa
pivot_acc = results_df.pivot(index='Model', columns='Fitur', values='Accuracy')
pivot_acc = pivot_acc[['MFCC', 'Pitch', 'MFCC+Pitch']]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.heatmap(pivot_acc, annot=True, fmt='.4f', cmap='YlGn', ax=axes[0],
            linewidths=1, linecolor='white', vmin=0.5, vmax=1.0)
axes[0].set_title('Heatmap Accuracy', fontsize=14, fontweight='bold')

pivot_f1 = results_df.pivot(index='Model', columns='Fitur', values='F1-Score')
pivot_f1 = pivot_f1[['MFCC', 'Pitch', 'MFCC+Pitch']]

sns.heatmap(pivot_f1, annot=True, fmt='.4f', cmap='YlOrRd', ax=axes[1],
            linewidths=1, linecolor='white', vmin=0.5, vmax=1.0)
axes[1].set_title('Heatmap F1-Score', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

# Analisis kontribusi pitch
print("=" * 60)
print("ANALISIS KONTRIBUSI PITCH")
print("=" * 60)
for m in model_names:
    sub = results_df[results_df['Model'] == m]
    acc_mfcc = sub[sub['Fitur'] == 'MFCC']['Accuracy'].values[0]
    acc_pitch = sub[sub['Fitur'] == 'Pitch']['Accuracy'].values[0]
    acc_both = sub[sub['Fitur'] == 'MFCC+Pitch']['Accuracy'].values[0]
    delta = acc_both - acc_mfcc
    print(f"\n{m}:")
    print(f"  MFCC only      : {acc_mfcc:.4f}")
    print(f"  Pitch only     : {acc_pitch:.4f}")
    print(f"  MFCC + Pitch   : {acc_both:.4f}")
    print(f"  Delta (MFCC+P vs MFCC): {delta:+.4f}")
    if acc_pitch > 0.7:
        print(f"  -> Pitch SAJA sudah memberikan akurasi {acc_pitch:.1%}, membuktikan pitch sangat diskriminatif.")
    if delta > 0:
        print(f"  -> Menambahkan Pitch ke MFCC MENINGKATKAN akurasi sebesar {delta:.4f}.")

## 10. Deep Learning: CNN pada Mel Spectrogram

Pada bagian ini kita akan:
1. Load file audio dari folder `dataset/`
2. Ekstrak Mel Spectrogram
3. Training CNN model
4. Bandingkan dengan ML Tradisional

In [ ]:
import librosa
from tqdm import tqdm
from prepocessing.voice_preprocessing_dl import preprocess_audio_dl
from features.feature_extractor_dl import extract_mel_spectrogram
from models.CNN_model import build_model, train_model as train_cnn, evaluate_model as eval_cnn

# Konfigurasi
DATASET_PATH = 'dataset'
TARGET_SR = 22050
FIXED_DURATION = 3.0
N_MELS = 128

# Kumpulkan file audio
audio_files = []
for label in ['male', 'female']:
    folder = os.path.join(DATASET_PATH, label)
    if os.path.exists(folder):
        for f in os.listdir(folder):
            if f.endswith(('.wav', '.mp3')):
                audio_files.append({'path': os.path.join(folder, f), 'label': label})

print(f"Total audio files: {len(audio_files)}")

In [ ]:
# Ekstraksi Mel Spectrogram
mel_specs = []
labels_dl = []

for item in tqdm(audio_files, desc="Extracting Mel Spectrograms"):
    y, sr = preprocess_audio_dl(item['path'], target_sr=TARGET_SR, fixed_duration=FIXED_DURATION)
    if y is not None:
        mel = extract_mel_spectrogram(y, sr, n_mels=N_MELS)
        mel_specs.append(mel)
        labels_dl.append(1 if item['label'] == 'male' else 0)

# Stack dan reshape
X_dl = np.array(mel_specs)
y_dl = np.array(labels_dl)

# Tambahkan channel dimension: (samples, n_mels, n_frames) -> (samples, n_mels, n_frames, 1)
X_dl = X_dl[..., np.newaxis]

print(f"Mel Spectrogram shape: {X_dl.shape}")
print(f"Labels shape: {y_dl.shape}")
print(f"Male: {np.sum(y_dl==1)}, Female: {np.sum(y_dl==0)}")

In [ ]:
# Train/Test Split untuk CNN
X_dl_train, X_dl_test, y_dl_train, y_dl_test = train_test_split(
    X_dl, y_dl, test_size=0.2, random_state=42, stratify=y_dl)

# Validation split dari training data
X_dl_train, X_dl_val, y_dl_train, y_dl_val = train_test_split(
    X_dl_train, y_dl_train, test_size=0.15, random_state=42, stratify=y_dl_train)

print(f"CNN Train : {X_dl_train.shape[0]}")
print(f"CNN Val   : {X_dl_val.shape[0]}")
print(f"CNN Test  : {X_dl_test.shape[0]}")

In [ ]:
# Build dan train CNN
input_shape = X_dl_train.shape[1:]  # (n_mels, n_frames, 1)
print(f"Input shape: {input_shape}")

cnn_model = build_model(input_shape)
cnn_model.summary()

In [ ]:
# Training
history = train_cnn(cnn_model, X_dl_train, y_dl_train,
                    X_dl_val, y_dl_val, epochs=50, batch_size=32)

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['loss'], label='Train Loss', color='#FF6B6B', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Val Loss', color='#4ECDC4', linewidth=2)
axes[0].set_title('Training & Validation Loss', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(history.history['accuracy'], label='Train Acc', color='#FF6B6B', linewidth=2)
axes[1].plot(history.history['val_accuracy'], label='Val Acc', color='#4ECDC4', linewidth=2)
axes[1].set_title('Training & Validation Accuracy', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Evaluasi CNN
cnn_results = eval_cnn(cnn_model, X_dl_test, y_dl_test)

print("=" * 50)
print("CNN EVALUATION RESULTS")
print("=" * 50)
print(f"Accuracy  : {cnn_results['accuracy']:.4f}")
print(f"Precision : {cnn_results['precision']:.4f}")
print(f"Recall    : {cnn_results['recall']:.4f}")
print(f"F1-Score  : {cnn_results['f1']:.4f}")
print(f"\n{cnn_results['classification_report']}")

## 11. Perbandingan Final: Tradisional ML vs Deep Learning

In [ ]:
# Tambahkan CNN ke tabel
cnn_row = {
    'Model': 'CNN (Mel Spectrogram)',
    'Fitur': 'Mel Spectrogram',
    'Accuracy': cnn_results['accuracy'],
    'Precision': cnn_results['precision'],
    'Recall': cnn_results['recall'],
    'F1-Score': cnn_results['f1'],
}
final_df = pd.concat([results_df, pd.DataFrame([cnn_row])], ignore_index=True)

# Sort by accuracy
final_df_sorted = final_df.sort_values('Accuracy', ascending=False).reset_index(drop=True)

styled_final = final_df_sorted.style.format({
    'Accuracy': '{:.4f}', 'Precision': '{:.4f}',
    'Recall': '{:.4f}', 'F1-Score': '{:.4f}'
}).background_gradient(subset=['Accuracy', 'F1-Score'], cmap='YlGn')

styled_final

In [ ]:
# Final bar chart
fig, ax = plt.subplots(figsize=(16, 7))

final_sorted = final_df_sorted.sort_values('Accuracy', ascending=True)
colors_final = ['#FF6B6B' if 'CNN' in m else '#4ECDC4' for m in final_sorted['Model']]
bar_labels = [f"{r['Model']}\n({r['Fitur']})" for _, r in final_sorted.iterrows()]

bars = ax.barh(range(len(final_sorted)), final_sorted['Accuracy'],
               color=colors_final, edgecolor='black', linewidth=0.5, height=0.6)

for i, (bar, acc) in enumerate(zip(bars, final_sorted['Accuracy'])):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f'{acc:.4f}', va='center', fontsize=10, fontweight='bold')

ax.set_yticks(range(len(final_sorted)))
ax.set_yticklabels(bar_labels, fontsize=10)
ax.set_xlabel('Accuracy', fontsize=13)
ax.set_title('Ranking Model: Tradisional ML vs Deep Learning (CNN)',
             fontsize=16, fontweight='bold')
ax.set_xlim(0, 1.15)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## 12. Kesimpulan

**Temuan Utama:**

1. **Pitch sebagai fitur krusial:** Secara empiris, Pitch saja sudah mampu memberikan akurasi yang signifikan dalam membedakan gender karena perbedaan fisiologis frekuensi dasar antara laki-laki (~85-180 Hz) dan perempuan (~165-255 Hz).

2. **MFCC + Pitch:** Kombinasi MFCC dan Pitch umumnya memberikan performa terbaik pada model tradisional, karena MFCC menangkap karakteristik spektral suara secara komprehensif, sementara Pitch memberikan informasi frekuensi dasar yang sangat diskriminatif.

3. **Tradisional ML vs CNN:** Model CNN pada Mel Spectrogram memberikan perspektif berbeda karena CNN dapat menangkap pola spasial dan temporal dari representasi visual suara secara otomatis tanpa perlu feature engineering manual.

4. **Rekomendasi:** Untuk deployment yang cepat dan ringan, model tradisional (terutama SVM atau Random Forest) dengan fitur MFCC+Pitch sudah sangat memadai. CNN lebih cocok jika data yang tersedia sangat besar dan komputasi bukan kendala.